# PhonePe Transaction Dispute Analyzer
**Industrial Project - Python & Pandas Data Pipeline**

### Problem Context
PhonePe processes thousands of daily digital transactions. Failed payments, pending refunds, and repeat complaints need to be investigated and prioritized.
This project consolidates 3 disparate datasets (`transactions.csv`, `refunds.xlsx`, `customer_complaints.json`) into a single master dispute dataset and classifies each dispute into High, Medium, or Low support priority.


---
## Setup: Imports & File Check
**Goal**: Import necessary libraries (`pandas`, `numpy`, `pathlib.Path`, `IPython.display.display`), verify the 3 data files exist, and create the `phonepe_reports` folder.


---
## Step 1 (Milestone 1): Load and Assess Transaction Data
**Goal**: Load `transactions.csv` and get an initial understanding of payment records.
- Load `transactions.csv` into `transactions_raw`.
- Display shape and first 5 rows.
- Inspect column data types (`dtypes`).
- Find unique counts for `transaction_id` and `customer_id`.
- Check missing values per column (`isna().sum()`).

*Expected Targets*:
- Shape: `(5032, 9)`
- Unique transactions: `5000`
- Missing amounts: `28`


---
## Step 2 (Milestone 2): Inspect Refunds and Complaints Data
**Goal**: Load and inspect the refunds (Excel) and customer complaints (JSON) datasets.
- Load `refunds.xlsx` into `refunds_raw`.
- Load `customer_complaints.json` into `complaints_raw`.
- Check shapes, first 3 rows, and missing values for both datasets.
- Check value counts of `refund_status` (look for case inconsistencies: 'Pending', 'pending', 'PENDING', etc.).
- Check value counts of `complaint_type`.

*Expected Targets*:
- Refunds shape: `(1575, 5)` (15 missing amounts)
- Complaints shape: `(1676, 6)` (0 missing values)


---
## Step 3 (Milestone 3): Clean Transaction Records
**Goal**: Clean and validate raw transaction data.
- Remove duplicate rows based on `transaction_id` into `transactions_clean`.
- Standardize `transaction_status`: strip whitespace and convert to Title Case.
- Clean `payment_mode`: strip whitespace.
- Convert `transaction_date` to datetime (`errors='coerce'`).
- Convert `amount` to numeric (`errors='coerce'`).
- Create boolean column `needs_data_review`: True if date is null, amount is null, or amount <= 0.

*Expected Targets*:
- Duplicate IDs removed: `32` (5,000 remaining)
- Status counts: Success: `3279`, Failed: `851`, Pending: `602`, Reversed: `268`
- Records requiring review: `62`


---
## Step 4 (Milestone 4): Clean Issue and Refund Records
**Goal**: Deduplicate refunds/complaints, standardize statuses, and verify referential integrity.
- Deduplicate `refunds_raw` on `refund_id` -> `refunds_clean`.
- Deduplicate `complaints_raw` on `complaint_id` -> `complaints_clean`.
- Standardize `refund_status` and `complaint_status` (strip and title case).
- Parse `refund_date`, `refund_amount`, and `complaint_date`.
- Check foreign keys: identify if any `transaction_id` does NOT exist in `transactions_clean`. Create boolean column `unmatched_transaction`.

*Expected Targets*:
- Unique refunds: `1550` (35 unmatched)
- Unique complaints: `1650` (30 unmatched)


---
## Step 5 (Milestone 5): Analyze Transaction Status and Payment Behavior
**Goal**: Identify failure and pending patterns across payment modes and customer cities.
- `status_summary`: Group by `transaction_status`, compute total count and sum of `amount`.
- `payment_mode_summary`: Group by `payment_mode` and `transaction_status`, compute counts.
- `city_issue_summary`: Identify top cities by count of Failed or Pending transactions.

*Expected Insights*:
- Highest volume & issue mode: UPI (600 Failed, 412 Pending).
- Top issue cities: Pune (167), Chennai (155), Hyderabad (153).


---
## Step 6 (Milestone 6): Connect Refund Requests to Transactions
**Goal**: Aggregate refund records per transaction and merge with clean transactions.
- In `refunds_clean`, create `refund_unresolved = refund_status.isin(['Pending', 'Failed'])`.
- Group refunds by `transaction_id` to compute:
  - `refund_request_count`: total refunds requested
  - `total_refund_requested`: sum of refund amounts
  - `unresolved_refund_count`: count of unresolved refunds
- Left join `transactions_clean` with the aggregated refund summary on `transaction_id` (`validate='one_to_one'`).
- Fill missing refund counts/amounts with `0`.

*Expected Targets*:
- Retained transactions: exactly `5000`
- Refunds status counts: Completed: `781`, Pending: `580`, Failed: `189`


---
## Step 7 (Milestone 7): Analyze Complaints and Repeat Issues
**Goal**: Aggregate customer complaints per transaction and identify repeated/unresolved complaints.
- In `complaints_clean`, create `complaint_unresolved = complaint_status.isin(['Open', 'In Progress'])`.
- Summarize `complaint_type` counts.
- Group complaints by `transaction_id` to compute:
  - `complaint_count`: total complaints for that transaction
  - `unresolved_complaint_count`: count of unresolved complaints
- Find repeat complaints (`complaint_count > 1`).
- Find top customers by complaint count.

*Expected Targets*:
- Status counts: Resolved: `628`, Open: `569`, In Progress: `453`
- Top transactions have up to 4 repeat complaints.


---
## Step 8 (Milestone 8): Build the Unified Dispute Master Table
**Goal**: Merge transactions, refunds, and complaints into a single consolidated DataFrame.
- Left join `transaction_refunds` with `complaint_agg` on `transaction_id` (`validate='one_to_one'`).
- Fill missing complaint metrics with `0` and convert count columns to integer.
- Validate with assertions:
  - `dispute_master['transaction_id'].is_unique`
  - `len(dispute_master) == 5000`
- Inspect shape (5000 rows, 15 columns).


---
## Step 9 (Milestone 9): Classify Dispute Support Priority
**Goal**: Classify each transaction dispute into High, Medium, or Low priority.
- Create 3 boolean issue signals:
  1. `payment_issue`: `transaction_status` is 'Failed' or 'Pending'
  2. `refund_issue`: `unresolved_refund_count > 0`
  3. `complaint_issue`: `complaint_count > 1` OR `unresolved_complaint_count > 0`
- Sum signals: `issue_signal_count = payment_issue + refund_issue + complaint_issue` (0 to 3)
- Assign `support_priority` using `np.select()`:
  - 'High' if `issue_signal_count >= 2`
  - 'Medium' if `issue_signal_count == 1`
  - 'Low' if `issue_signal_count == 0` (default)
- Generate `priority_summary` count table.

*Expected Targets*:
- High: `535`
- Medium: `1998`
- Low: `2467`


---
## Step 10 (Milestone 10): Export Dispute Resolution Reports and Present Findings
**Goal**: Sort, export CSV reports, and summarize executive findings.
- Sort `dispute_master` by `support_priority` (High -> Medium -> Low), then `transaction_id`.
- Export to `phonepe_reports/phonepe_transaction_dispute_report.csv`.
- Export `priority_summary` to `phonepe_reports/phonepe_dispute_priority_summary.csv`.
- Print executive metrics:
  - Failed or pending transactions: `1453`
  - Transactions with unresolved refunds: `699`
  - Transactions with complaint signals: `949`
  - High-priority transactions: `535`
